In [1]:
import pandas as pd

# Load train and test datasets from the project's raw_data folder
train = pd.read_csv('raw_data/train.csv')
test = pd.read_csv('raw_data/test.csv')

# Check the dimensions of both datasets (rows, columns)
print(train.shape)
print(test.shape)

# Preview the first 5 rows of the training set
train.head()


FileNotFoundError: [Errno 2] No such file or directory: 'raw_data/train.csv'

In [ ]:
# Check column names and data types
print(train.dtypes)

In [ ]:
# Check for missing values in each column
train.isnull().sum().sort_values(ascending=False)

In [ ]:
# Summary statistics for all numeric columns
train.describe()

In [ ]:
import matplotlib.pyplot as plt

# Distribution of our target variable
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Histogram
axes[0].hist(train['actual_finish_time_minutes'].dropna(), bins=50, edgecolor='white', color='steelblue')
axes[0].set_title('Finish Time Distribution')
axes[0].set_xlabel('Minutes')
axes[0].set_ylabel('Count')

# Boxplot to spot outliers
axes[1].boxplot(train['actual_finish_time_minutes'].dropna(), vert=False)
axes[1].set_title('Finish Time Boxplot')
axes[1].set_xlabel('Minutes')

plt.tight_layout()
plt.show()

# Key stats
print(train['actual_finish_time_minutes'].describe())


In [ ]:
# Check all categorical columns and their unique values
cat_cols = train.select_dtypes(include='object').columns.tolist()

for col in cat_cols:
    print(f"\n{col} ({train[col].nunique()} unique):")
    print(train[col].value_counts())
    

In [ ]:
# Correlation of all numeric features with the target
corr = (train
        .select_dtypes(include='number')
        .corr()['actual_finish_time_minutes']
        .drop('actual_finish_time_minutes')
        .sort_values())

# Plot as horizontal bar chart
fig, ax = plt.subplots(figsize=(10, 10))
colors = ['steelblue' if v < 0 else 'tomato' for v in corr]
corr.plot(kind='barh', ax=ax, color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Feature Correlation with Finish Time', fontsize=14)
ax.set_xlabel('Pearson Correlation')
plt.tight_layout()
plt.show()

# Print top 5 positive and negative correlates
print("🔴 Top 5 features that INCREASE finish time (slower):")
print(corr.tail(5))
print("\n🔵 Top 5 features that DECREASE finish time (faster):")
print(corr.head(5))

In [ ]:
# Boxplots of finish time by categorical features
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Finish time by training program (ordinal)
order = ['Beginner', 'Intermediate', 'Advanced']
for i, (col, ax) in enumerate(zip(
    ['training_program', 'gender', 'marathon_weather', 'course_difficulty'],
    axes.flatten()
)):
    groups = [train[train[col] == cat]['actual_finish_time_minutes'].dropna()
              for cat in train[col].unique()]
    labels = train[col].unique()
    ax.boxplot(groups, labels=labels, vert=True)
    ax.set_title(f'Finish Time by {col}')
    ax.set_ylabel('Minutes')
    ax.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

In [ ]:
# Scatter plots of the strongest numeric predictors vs finish time
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

pairs = [
    ('running_experience_months', 'Experience (months)'),
    ('speed_work_sessions_per_week', 'Speed Sessions/Week'),
    ('runs_per_week', 'Runs Per Week'),
    ('rest_days_per_week', 'Rest Days/Week'),
    ('resting_heart_rate_bpm', 'Resting Heart Rate'),
    ('vo2_max', 'VO2 Max'),
]

for ax, (col, label) in zip(axes.flatten(), pairs):
    ax.scatter(train[col], train['actual_finish_time_minutes'],
               alpha=0.1, s=5, color='steelblue')
    ax.set_xlabel(label)
    ax.set_ylabel('Finish Time (min)')
    ax.set_title(f'{label} vs Finish Time')

plt.tight_layout()
plt.show()


In [ ]:
import seaborn as sns

# Pairplot of top numeric predictors
cols = ['running_experience_months', 'speed_work_sessions_per_week',
        'runs_per_week', 'vo2_max', 'resting_heart_rate_bpm', 'actual_finish_time_minutes']

sns.pairplot(train[cols].dropna(), plot_kws={'alpha': 0.1, 's': 5})
plt.show()

In [ ]:
# Does gender interact with training program?
sns.boxplot(data=train, x='training_program', y='actual_finish_time_minutes',
            hue='gender', order=['Beginner', 'Intermediate', 'Advanced'])
plt.title('Finish Time by Training Program and Gender')
plt.show()

In [ ]:
# Profile of runners who DNF'd vs finished
train['dnf'] = train['actual_finish_time_minutes'].isna().astype(int)

print(train.groupby('dnf')[['age', 'running_experience_months',
      'injury_count', 'weekly_mileage_km','sleep_hours_avg']].mean())

In [ ]:
# Extract month and look at average finish time per month
train['month'] = pd.to_datetime(train['marathon_date']).dt.month

train.groupby('month')['actual_finish_time_minutes'].mean().plot(
    kind='bar', color='steelblue', edgecolor='white')
plt.title('Average Finish Time by Race Month')
plt.xlabel('Month')
plt.ylabel('Avg Finish Time (min)')
plt.show()

In [ ]:
# Fill NaN as 'None' temporarily and compare finish times
train['injury_severity_filled'] = train['injury_severity'].fillna('None')

order = ['None', 'Minor', 'Moderate', 'Severe']
sns.boxplot(data=train, x='injury_severity_filled',
            y='actual_finish_time_minutes', order=order)
plt.title('Finish Time by Injury Severity')
plt.show()


In [ ]:
# ============================================================
# EDA FINDINGS SUMMARY — based on confirmed outputs
# ============================================================

findings = """
=== DATASET OVERVIEW ===
- 80,000 runners in train, 20,000 in test, 41 features
- Target: actual_finish_time_minutes (range 171–400 min, mean ~288 min / 4h48)
- 1,989 missing actual_finish_time_minutes → DNFs (did not finish)

=== MISSING VALUES ===
- injury_severity: 47,350 missing (59%) → NaN means no injury, recode as 'None'
- personal_best_minutes: 10,745 missing → likely first-time marathoners
- vo2_max: 9,611 missing → not everyone gets lab tested
- cross_training, nutrition_score, hydration_consistency: moderate missingness (self-reported)
- sleep_hours_avg: 3,942 missing

=== CATEGORICAL FEATURES ===
- gender: Male ~48% / Female ~48% / Non-binary ~4%
  → Weak predictor — almost no difference in finish time across genders
- training_program: Beginner (50%) / Intermediate (32%) / Advanced (18%)
  → Strong ordinal predictor: Beginner ~310 / Intermediate ~280 / Advanced ~245 min
- marathon_weather: 6 categories, fairly balanced
  → Hot weather shows slightly higher finish times, minimal difference overall
- course_difficulty: Flat / Mixed / Hilly fairly balanced
  → Hilly courses show noticeably higher finish times than Flat
- marathon_date: finish time flat across all 12 months (~288 min)
  → Month/season is NOT a useful feature → drop it

=== NUMERIC FEATURES ===
- running_experience_months: strongest legitimate predictor (r=-0.65)
  → Strong non-linear negative relationship, most gains from 0–50 months
- speed_work_sessions_per_week (r=-0.35): more sessions = faster, clear step pattern
- runs_per_week (r=-0.31): more runs = faster, clear step pattern
- rest_days_per_week (r=+0.30): more rest = slower
- resting_heart_rate_bpm (r=+0.20): higher HR = slightly slower, noisy
- vo2_max (r=-0.35): higher VO2 max = meaningfully faster

=== INJURY SEVERITY ===
- Clear ordinal effect: None ~285 / Minor ~293 / Moderate ~305 / Severe ~310 min
- Encode ordinally as: None=0, Minor=1, Moderate=2, Severe=3

=== DNF ANALYSIS ===
- DNF runners are older (+2.4 yrs), less experienced (-8 months), more injuries (1.12 vs 0.54)
- Implication: drop DNFs from training set or model separately

=== LEAKY FEATURES — exclude from model ===
- target_finish_time_minutes (r=0.95) — runner's own pre-race goal
- personal_best_minutes (r=0.86) — use carefully, may leak
- medal_outcome (r=-0.38) — race outcome, not a predictor
- weekly_mileage_miles — exact duplicate of weekly_mileage_km

=== PREPROCESSING PLAN ===
1. Recode injury_severity NaN → 'None', encode ordinally (0–3)
2. Flag first-time marathoners where personal_best_minutes is NaN
3. Encode training_program ordinally (Beginner=0, Intermediate=1, Advanced=2)
4. OHE: marathon_weather, course_difficulty, gender
5. Drop: marathon_date, weekly_mileage_miles, target_finish_time_minutes, medal_outcome
6. Drop DNF rows (actual_finish_time_minutes is NaN)
7. Build baseline model (Linear Regression)
"""

print(findings)

In [ ]:
# Copy
train_clean = train.copy()
test_clean = test.copy()

In [ ]:
# Step 1 — Drop DNF rows (no finish time = can't train on them)
print(f"Before: {train_clean.shape}")
train_clean = train_clean.dropna(subset=['actual_finish_time_minutes'])
print(f"After dropping DNFs: {train_clean.shape}")

In [ ]:
# Check remaining missing values after dropping DNFs
train_clean.isnull().sum().sort_values(ascending=False)

In [ ]:
# Full list of features in the dataset
print(list(train_clean.columns))


In [ ]:
# Full list of features in a readable format
for i, col in enumerate(train_clean.columns, 1):
    print(f"{i:02d}. {col}")
    

In [ ]:
!git add .
!git commit -m "My Notebook"
!git push origin HEAD:Francesco

In [ ]:
training_map = {"Beginner": 1, "Intermediate": 2, "Advanced": 3}
course_map = {"Flat": 1, "Mixed": 2, "Hilly": 3}
injury_map = {"Minor": 1, "Moderate": 2, "Severe": 3}

In [ ]:
df['training_program_enc'] = df['training_program'].map(training_map)
df['course_difficulty_enc'] = df['course_difficulty'].map(course_map)
